In [ ]:
%pip install --quiet --upgrade langchain-text-splitters langchain-community faiss-cpu
%pip install -qU "langchain[openai]"
!pip install langchain-google-genai
%pip install --upgrade google-ai-generativelanguage>=0.6.18,<0.7.0 langchain-google-genai

In [ ]:
import os
import bs4
from bs4 import BeautifulSoup
import re
import pandas as pd
import csv
import json
import requests
import html
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import google.generativeai as genai
from datetime import datetime
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter
from langchain.schema import Document
from typing import List, TypedDict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.environ["USER_AGENT"] = "Mozilla/5.0 (compatible; MyLangChainBot/1.0; +http://mywebsite.com/bot)"
os.environ["LANGCHAIN_TRACING_V2"]="false"
os.environ["LANGCHAIN_API_KEY"]="YOUR_LANGCHAIN_API_KEY"
os.environ["OPENAI_API_KEY"]="YOUR_OPENAI_API_KEY"
os.environ["MISTRAL_AI_API_KEY"]="YOUR_MISTRAL_API_KEY"
os.environ["GEMINI_API_KEY"]="YOUR_GEMINI_API_KEY"
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

CSV_PROGRESS = "/content/drive/MyDrive/Colab Notebooks/mod_progress.csv"
CSV_MESSAGES = "/content/drive/MyDrive/Colab Notebooks/mod_progress_messages.csv"
CSV_PROGRESS_OUTPUT = "/content/drive/MyDrive/Colab Notebooks/mod_progress_output.csv"
CSV_MESSAGES_OUTPUT = "/content/drive/MyDrive/Colab Notebooks/mod_progress_messages_output.csv"
FAISS_INDEX = "/content/drive/MyDrive/Colab Notebooks/faiss_index"
CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/Stats/results_summary.csv"
CSV_CHUNK = "/content/drive/MyDrive/Colab Notebooks/Stats/chunk_result_summary.csv"
JSON_OUTPUT = "/content/drive/MyDrive/Colab Notebooks/output.json"
ENABLE_CHUNK_TEST = False

In [ ]:
# Functions

# Define clear function
def clearDocs(raw_html):
  soup = BeautifulSoup(str(raw_html), "html.parser")
  text = soup.get_text()
  text = re.sub(r"http\S+\www\S+", "", text) # links
  text = re.sub(r"[^\x00-\x7F]+", "", text)
  text = re.sub(r"\s+", " ", text)
  return text.strip()

# Prompt function
def build_prompt_template(response_type="default"):
    if response_type == "boolean":
        template = (
            "Rispondi alla seguente domanda in italiano con 'True' o 'False', "
            "usando solo il contesto fornito. Rispondi comunque True o False, "
            "anche se non sei sicuro.\n\n"
            "Contesto:\n{context}\n\nDomanda:\n{question}\n\nRisposta:"
        )
    else:
        template = (
            "Rispondi alla seguente domanda in italiano, utilizzando solo le informazioni fornite nel contesto. "
            "Se non sei sicuro, fornisci comunque la risposta più probabile in una frase breve.\n\n"
            "Contesto:\n{context}\n\nDomanda:\n{question}\n\nRisposta:"
        )
    return PromptTemplate.from_template(template)

#Define print function
def printQA(question, answer):
    print(f"Domanda:\n{question}\n")
    print("\n" + "=" * 50 + "\n")
    print("Contesto trovato:\n")
    print(docs_content)
    print("\n" + "=" * 50 + "\n")
    print("Risposta:\n")
    print(answer.content)
    print("\n" + "=" * 50 + "\n")

# Comparison function
def compare_answers(generated, correct):
    return generated.strip().lower() == correct.strip().lower()

# % of success
def calculate_success_rate(generated_answers, correct_answers):
    assert len(generated_answers) == len(correct_answers), "Numero di risposte deve coincidere"
    matches = sum(compare_answers(gen, cor) for gen, cor in zip(generated_answers, correct_answers))
    return matches / len(correct_answers) * 100

# Reading file function
def read_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f.readlines() if line.strip()]

# Semantic similarity function
def semantic_similarity(answer1, answer2):
    vec1 = embedding_model.embed_query(answer1)
    vec2 = embedding_model.embed_query(answer2)
    return cosine_similarity([vec1], [vec2])[0][0]

# Variables
llm_models = {
    # "gpt-4.1": ChatOpenAI(model="gpt-4.1", temperature=0.4),
    # "gpt-4-turbo": ChatOpenAI(model="gpt-4-turbo", temperature=0.4),
    # "gpt-4": ChatOpenAI(model="gpt-4-turbo", temperature=0.4),
    # "gpt-3.5-turbo": ChatOpenAI(model="gpt-3.5-turbo", temperature=0.4),
    "gemini-2.5-flash": ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.4,
        api_key=os.environ["GEMINI_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai"
       ),
    "mistral-tiny": ChatOpenAI(
        model="mistral-tiny",
        temperature=0.4,
        api_key=os.environ["MISTRAL_AI_API_KEY"],
        base_url="https://api.mistral.ai/v1",
    ),
}

chunk_configs = [
    {"chunk_size": 800, "chunk_overlap": 100},
    {"chunk_size": 500, "chunk_overlap": 150},
    {"chunk_size": 300, "chunk_overlap": 50},
]

test_sets = {
    "Test1": {
        "questions": "/content/drive/MyDrive/Colab Notebooks/Stats/QUESTION/Question1.txt",
        "expected": "/content/drive/MyDrive/Colab Notebooks/Stats/ANSWER/ExpectedAnswer1.txt",
    },
    "Test2": {
        "questions": "/content/drive/MyDrive/Colab Notebooks/Stats/QUESTION/Question2.txt",
        "expected": "/content/drive/MyDrive/Colab Notebooks/Stats/ANSWER/ExpectedAnswer2.txt",
    },
    "Test3": {
        "questions": "/content/drive/MyDrive/Colab Notebooks/Stats/QUESTION/Question3.txt",
        "expected": "/content/drive/MyDrive/Colab Notebooks/Stats/ANSWER/ExpectedAnswer3.txt",
    }
}

test_sets_1 = {
    "Test4": {
        "questions": "/content/drive/MyDrive/Colab Notebooks/Stats/QUESTION/Question4.txt",
        "expected": "/content/drive/MyDrive/Colab Notebooks/Stats/ANSWER/ExpectedAnswer4.txt",
    },
    "Test5": {
        "questions": "/content/drive/MyDrive/Colab Notebooks/Stats/QUESTION/Question5.txt",
        "expected": "/content/drive/MyDrive/Colab Notebooks/Stats/ANSWER/ExpectedAnswer5.txt",
    }
}


prompt = build_prompt_template("boolean")

# Function which extracts generated answers
def extract_generated_answers(file_path):
    answers = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if "RISPOSTA GENERATA:" in line:
                answers.append(line.split("RISPOSTA GENERATA:")[1].strip())
    return answers

In [ ]:
# Loading and cleaning csv files

# CSV progress messages
if os.path.exists(CSV_MESSAGES_OUTPUT):
    print("File mod_progress_output.csv già esistente. Carico quello.\n")
    msg = pd.read_csv(CSV_MESSAGES_OUTPUT)
else:
    print("File pulito non trovato. Procedo con la pulizia.\n")
    msg = pd.read_csv(CSV_MESSAGES)

    msg["clean_text"] = msg["tText"].apply(clearDocs)
    msg = msg[msg["clean_text"].str.len() > 15]
    msg = msg.drop(columns=["tText"])

    msg = msg.rename(columns={
    "iId": "msg_id",
    "iProgess": "progress_id",
    "iFather": "answer_to_msg_id",
    "clean_text": "content",
    "dDate": "timestamp",
    "sAttach": "attachment",
    "iAuthor": "author"
    })[["msg_id", "progress_id", "answer_to_msg_id", "content", "timestamp", "attachment", "author"]]

    msg.to_csv(CSV_MESSAGES_OUTPUT, index=False, encoding="utf-8", errors="replace")

# CSV progress
if os.path.exists(CSV_PROGRESS_OUTPUT):
    print("File mod_progress_messages_output.csv già esistente. Carico quello.\n")
    pro = pd.read_csv(CSV_PROGRESS_OUTPUT)
else:
    print("File pulito non trovato. Procedo con la pulizia.\n")
    pro = pd.read_csv(CSV_PROGRESS)

    pro["clean_desc"] = pro["sDescription"].apply(clearDocs)
    pro = pro.drop(columns=["sDescription"])

    pro = pro.rename(columns={
    "iId": "progress_id",
    "sTitle": "subject",
    "clean_desc": "description",
    "dOpen": "created_at",
    "dClose": "closed_at",
    "dLastChange": "updated_at",
    "iAuthor": "author"
    })[["progress_id", "subject", "description", "created_at", "closed_at", "updated_at", "author"]]

    pro.to_csv(CSV_PROGRESS_OUTPUT, index=False, encoding="utf-8", errors="replace")

In [ ]:
# JSON progress output
if os.path.exists(JSON_OUTPUT):
    print("File output.json già esistente. Carico quello.\n")
else:
    print("File non trovato. Procedo a crearlo.\n")
    grouped = msg.groupby("progress_id")
    dataset = []
    num_removed = 0
    skipped_progress_empty_metadata = 0

    for progress_id, group in grouped:
        messages = group.sort_values("timestamp").to_dict(orient="records")
        texts = [m["content"] for m in messages]
        if len(texts) > 1:
          tfidf = TfidfVectorizer().fit_transform(texts)
          sim_matrix = cosine_similarity(tfidf)
          to_remove = set()
          for i in range(len(messages)):
            for j in range(i+1, len(messages)):
              if sim_matrix[i, j] >= 0.95:
                to_remove.add(j)

          num_removed += len(to_remove)
          messages = [m for idx, m in enumerate(messages) if idx not in to_remove]

          unique_authors = set(m["author"] for m in messages)
          total_length = sum(len(m["content"]) for m in messages)
          progress_info = pro[pro["progress_id"] == progress_id]
          if progress_info.empty:
            skipped_progress_empty_metadata += 1
            continue
          meta = progress_info.iloc[0].to_dict()

          dataset.append({
            "progress_id": progress_id,
            "subject": meta["subject"],
            "description": meta["description"],
            "created_at": meta["created_at"],
            "closed_at": meta["closed_at"],
            "updated_at": meta["updated_at"],
            "author": meta["author"],
            "message_count": len(messages),
            "total_char_length": total_length,
            "distinct_authors": len(unique_authors),
            "messages": messages
          })

          dataset = [p for p in dataset if p["message_count"] > 0]
    # Save output.json
    with open(JSON_OUTPUT, "w", encoding="utf-8", errors="replace") as f:
      json.dump(dataset, f, ensure_ascii=False, indent=2)

# Grouping content by progress_id
grouped_msg = msg.groupby("progress_id")["content"].apply(lambda x: "\n".join(x)).reset_index()
grouped = pd.merge(grouped_msg, pro, on="progress_id", how="left")

grouped["full_text"] = (
    grouped["subject"].fillna("") + "\n\n" +
    grouped["description"].fillna("") + "\n\n" +
    grouped["content"].fillna("")
)

In [ ]:
# Dataset analysis
with open(JSON_OUTPUT, "r", encoding="utf-8") as f:
    dataset = json.load(f)

progress_analysis = pd.DataFrame([{
    "progress_id": p["progress_id"],
    "closed_at": p.get("closed_at", None),
    "author": p["author"],
    "message_count": p["message_count"],} for p in dataset])

flat_mes = []
for progress in dataset:
  for m in progress["messages"]:
    flat_mes.append({
        "progress_id": progress["progress_id"],
        "mes_content": m["content"],
        "progress_author": m["author"],
    })

mes_analysis = pd.DataFrame(flat_mes)
mes_analysis["length"] = mes_analysis["mes_content"].str.len()
mes_analysis["word_count"] = mes_analysis["mes_content"].str.split().str.len()

# Total number of progress & messages
n_pro = progress_analysis.shape[0]
print("Numero di progress: ", n_pro)
n_mes = mes_analysis.shape[0]
print("Numero di messaggi: ", n_mes)

# Number of messages per progress
n_mes_thread = progress_analysis["message_count"]
print("\nNumero di messaggi per ogni progress: ")
print(n_mes_thread.describe())

# Mean of messages per progress (characters)
mean_length_progress = mes_analysis.groupby("progress_id")["length"].mean()
print("\nLunghezza media dei messaggi per ogni progress: ")
print(mean_length_progress.describe())

# Mean of messages per progress (words)
mean_words_progress = mes_analysis.groupby("progress_id")["word_count"].mean()
print("\nLunghezza media in parole dei messaggi per ogni progress:")
print(mean_words_progress.describe())

# Number of authors
n_authors = mes_analysis["progress_author"].nunique()
print("\nNumero di autori: ", n_authors)

# Mean of messages per author per progress
n_mes_author = mes_analysis.groupby(["progress_id", "progress_author"]).size().reset_index(name="count")
mean_mes_author = n_mes_author.groupby("progress_id")["count"].mean()
print("\nMedia di messaggi per autore per thread:")
print(mean_mes_author.describe())

# Number of open/close progress
progress_analysis["closed_at"] = pd.to_datetime(progress_analysis["closed_at"], errors="coerce")
n_closed = progress_analysis["closed_at"].notna().sum()
n_open = progress_analysis["closed_at"].isna().sum()
print(f"\nThread chiusi: {n_closed}")
print(f"Thread aperti: {n_open}")
print("-" * 80)

# Histograms
# Histogram of number of message per progress
sns.histplot(np.log(n_mes_thread), bins=50, kde=False)
plt.title("Numero di messaggi per ogni progress")
plt.xlabel("log(messaggi)")
plt.ylabel("Numero di thread")
plt.show()

# Histogram of the length of messages
sns.histplot(np.log(mes_analysis["length"]), bins=50, kde=False)
plt.title("Distribuzione della lunghezza dei messaggi")
plt.xlabel("log(caratteri)")
plt.ylabel("Frequenza")
plt.show()

# Histogram of the mean of messages per author per progress
sns.histplot(mean_mes_author, bins=50, kde=False)
plt.title("Distribuzione media messaggi per autore per thread")
plt.xlabel("Media messaggi per autore")
plt.ylabel("Numero di thread")
plt.show()

In [ ]:
# Splitting documents
with open(JSON_OUTPUT, "r", encoding="utf-8") as f:
    dataset = json.load(f)

docs = []
for progress in dataset:
    header = f"{progress['subject']}\n\n{progress['description']}"
    message_texts = "\n".join(m["content"] for m in progress["messages"])
    full_text = f"{header}\n\n{message_texts}"

    docs.append(Document(
        page_content=full_text,
        metadata={"progress_id": progress["progress_id"]}
    ))

# Test for different chunk_size & chunk_overlap
if ENABLE_CHUNK_TEST:
    # Result chunk tests
    chunk_test_results = []

    for config in chunk_configs:
        chunk_size = config["chunk_size"]
        chunk_overlap = config["chunk_overlap"]

        splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        chunks = splitter.split_documents(docs)
        chunks = [chunk for chunk in chunks if len(chunk.page_content.strip()) > 40]

        embedding_model = OpenAIEmbeddings()
        vector_store = FAISS.from_documents(chunks, embedding_model)

        for test_name, paths in test_sets.items():
            questions = read_lines(paths["questions"])
            expected_answers = read_lines(paths["expected"])
            for model_name, llm in llm_models.items():
                generated_answers = []
                qa_output_path = f"/content/drive/MyDrive/Colab Notebooks/Stats/TEST_RESULTS/QA_{test_name}_{model_name}.txt"

                with open(qa_output_path, "w", encoding="utf-8") as f:
                    for idx, (question, expected) in enumerate(zip(questions, expected_answers), start=1):
                        retrieved_docs = vector_store.similarity_search(question)
                        docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

                        message = prompt.invoke({"question": question, "context": docs_content})
                        response = llm.invoke(message)
                        generated = response.content.strip()

                        generated_answers.append(generated)

                        f.write(f"{idx}. DOMANDA: {question}\n")
                        f.write(f"   RISPOSTA ATTESA: {expected}\n")
                        f.write(f"   RISPOSTA GENERATA: {generated}\n")
                        f.write("-" * 80 + "\n")

                # % of success
                success = calculate_success_rate(generated_answers, expected_answers)

                chunk_test_results.append({
                    "test": test_name,
                    "model": model_name,
                    "chunk_size": chunk_size,
                    "chunk_overlap": chunk_overlap,
                    "success": success
                })

        correct_answers = read_lines(paths["expected"])

        success = calculate_success_rate(generated_answers, correct_answers)
        retrieval_strategy = "similarity"

        writer_header = not os.path.exists(CSV_CHUNK)

        # Creating a table of chunk_result_summary
        with open(CSV_CHUNK, mode="a", newline='', encoding="utf-8") as csv_file:
            writer = csv.writer(csv_file)
            if writer_header:
              writer.writerow(["Test", "Model", "Chunk Size", "Chunk Overlap", "Successo (%)"])

            for result in chunk_test_results:
                writer.writerow([
                    result["test"],
                    result["model"],
                    result["chunk_size"],
                    result["chunk_overlap"],
                    f"{result['success']:.2f}"
                ])
else:
    # Normal process
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(docs)
    chunks = [chunk for chunk in chunks if len(chunk.page_content.strip()) > 40]

    # Embedding and FAISS fase
    embedding_model = OpenAIEmbeddings()
    if os.path.exists(FAISS_INDEX):
        print("Vector store già esistente. Carico quello.\n")
        vector_store = FAISS.load_local(FAISS_INDEX, embedding_model, allow_dangerous_deserialization=True)
    else:
        print("Vector store non trovato. Procedo con la creazione.\n")
        vector_store = FAISS.from_documents(chunks, embedding_model)
        # Saving the vector store
        vector_store.save_local(FAISS_INDEX)

In [ ]:
# Prompt and query
question = "Nel 2024 sono state attivate 5 piattaforme per il whistleblowing?"

retrieved_docs = vector_store.similarity_search(question)
docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

message = prompt.invoke({"question": question, "context": docs_content })
answer = llm_models["gpt-4.1"].invoke(message)

printQA(question, answer)
print("Risposta attesa: " + "\n")
print("True")

In [ ]:
# Piece of code where there are some user's questions and expected answers (TRUE or FALSE)
# RAG system now tries to reply at user's questions
generated_answers = []
# Result summary for the % of success
results_summary = []

for test_name, paths in test_sets.items():
    questions = read_lines(paths["questions"])
    expected_answers = read_lines(paths["expected"])
    for model_name, llm in llm_models.items():
        generated_answers = []
        qa_output_path = f"/content/drive/MyDrive/Colab Notebooks/Stats/TEST_RESULTS_SIMILARITY/TRUE_FALSE/QA_{test_name}_{model_name}.txt"

        with open(qa_output_path, "w", encoding="utf-8") as f:
            for idx, (question, expected) in enumerate(zip(questions, expected_answers), start=1):
                retrieved_docs = vector_store.similarity_search(question)
                docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
                message = prompt.invoke({"question": question, "context": docs_content})

                if "gemini" in model_name.lower():
                    success = False
                    for attempt in range(5):
                        try:
                            response = llm.invoke(message)
                            success = True
                            break
                        except Exception as e:
                            if "429" in str(e) or "ResourceExhausted" in str(e):
                                wait_time = 6
                                print(f"Quota superata, attendo {wait_time}s...")
                                time.sleep(wait_time)
                            else:
                                raise e

                    if not success:
                        print("Richiesta fallita dopo i retry")
                        continue

                    time.sleep(6)

                else:
                    response = llm.invoke(message)

                generated = response.content.strip()
                generated_answers.append(generated)

                f.write(f"{idx}. DOMANDA: {question}\n")
                f.write(f"   RISPOSTA ATTESA: {expected}\n")
                f.write(f"   RISPOSTA GENERATA: {generated}\n")
                f.write("-" * 80 + "\n")

        # % of success
        success = calculate_success_rate(generated_answers, expected_answers)

        results_summary.append({
            "test": test_name,
            "model": model_name,
            "success": success
        })

correct_answers = read_lines(paths["expected"])

success = calculate_success_rate(generated_answers, correct_answers)
retrieval_strategy = "similarity"

# Creating the table of results_summary
with open(CSV_PATH, mode="a", newline='', encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)

    if csv_file.tell() == 0:
        writer.writerow(["Timestamp", "Test", "Modello", "Successo (%)", "Retrieval Strategy"])

    for result in results_summary:
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            result["test"],
            result["model"],
            f"{result['success']:.2f}",
            retrieval_strategy
        ])

In [ ]:
# Piece of code where there are some user's questions and expected answers (FREE ANSWERS)
prompt = build_prompt_template()
# RAG system now tries to reply at user's questions
generated_answers = []
# Result summary for the % of success
results_summary = []

for test_name, paths in test_sets_1.items():
    questions = read_lines(paths["questions"])
    expected_answers = read_lines(paths["expected"])
    for model_name, llm in llm_models.items():
        generated_answers = []
        qa_output_path = f"/content/drive/MyDrive/Colab Notebooks/Stats/TEST_RESULTS_SIMILARITY/DOMANDE_APERTE_09/QA_{test_name}_{model_name}.txt"

        with open(qa_output_path, "w", encoding="utf-8") as f:
            for idx, (question, expected) in enumerate(zip(questions, expected_answers), start=1):
                retrieved_docs = vector_store.similarity_search(question)
                docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
                message = prompt.invoke({"question": question, "context": docs_content})

                if "gemini" in model_name.lower():
                    success = False
                    for attempt in range(5):
                        try:
                            response = llm.invoke(message)
                            success = True
                            break
                        except Exception as e:
                            if "429" in str(e) or "ResourceExhausted" in str(e):
                                wait_time = 6
                                print(f"[{model_name}] Quota superata, attendo {wait_time}s...")
                                time.sleep(wait_time)
                            else:
                                raise e

                    if not success:
                        print(f"[{model_name}] Richiesta fallita dopo i retry")
                        continue

                    time.sleep(6)

                else:
                    response = llm.invoke(message)

                generated = response.content.strip()
                generated_answers.append(generated)

                similarity = semantic_similarity(generated, expected)
                threshold = 0.9
                correct = similarity >= threshold

                f.write(f"{idx}. DOMANDA: {question}\n")
                f.write(f"   RISPOSTA ATTESA: {expected}\n")
                f.write(f"   RISPOSTA GENERATA: {generated}\n")
                f.write(f"   SIMILARITÀ SEMANTICA: {similarity:.2f}\n")
                f.write(f"   CORRETTA? {'✅' if correct else '❌'}\n")
                f.write("-" * 80 + "\n")

In [ ]:
# Results of QA
CSV_SUMMARY="/content/drive/MyDrive/Colab Notebooks/Stats/tab_stats.csv"

# Folders
TRUE_FALSE = "/content/drive/MyDrive/Colab Notebooks/Stats/TEST_RESULTS_SIMILARITY/TRUE_FALSE"
DOMANDE_APERTE = "/content/drive/MyDrive/Colab Notebooks/Stats/TEST_RESULTS_SIMILARITY"

thresholds = [0.5, 0.75, 0.9]

results_tab_stats = []

# TRUE/FALSE
for filename in os.listdir(TRUE_FALSE):
    if not filename.endswith(".txt"):
        continue

    name = filename.replace("QA_", "").replace(".txt", "")
    test_name, model_name = name.split("_", 1)

    file_path = os.path.join(TRUE_FALSE, filename)
    generated_answers = extract_generated_answers(file_path)
    expected_answers = read_lines(test_sets[test_name]["expected"])

    success = calculate_success_rate(generated_answers, expected_answers)

    results_tab_stats.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "test": test_name,
        "model": model_name,
        "type": "TRUE/FALSE",
        "threshold": "",
        "success": success,
        "retrieval": "similarity"
    })

# DOMANDE_APERTE
for threshold in thresholds:
    folder = f"DOMANDE_APERTE_{str(threshold).replace('.', '')}"
    folder_path = os.path.join(DOMANDE_APERTE, folder)

    for filename in os.listdir(folder_path):
        if not filename.endswith(".txt"):
            continue

        name = filename.replace("QA_", "").replace(".txt", "")
        test_name, model_name = name.split("_", 1)

        file_path = os.path.join(folder_path, filename)
        generated_answers = extract_generated_answers(file_path)
        expected_answers = read_lines(test_sets_1[test_name]["expected"])

        correct_count = sum(
            1 for g, e in zip(generated_answers, expected_answers)
            if semantic_similarity(g, e) >= threshold
        )
        success = (correct_count / len(expected_answers)) * 100

        results_tab_stats.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "test": test_name,
            "model": model_name,
            "type": "DOMANDE_APERTE",
            "threshold": threshold,
            "success": success,
            "retrieval": "similarity"
        })

with open(CSV_SUMMARY, mode="w", newline='', encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "Timestamp", "Test", "Modello", "Tipo Domande",
        "Threshold", "Successo (%)", "Retrieval Strategy"
    ])
    for result in results_tab_stats:
        writer.writerow([
            result["timestamp"],
            result["test"],
            result["model"],
            result["type"],
            result["threshold"],
            f"{result['success']:.2f}",
            result["retrieval"]
        ])

print(f"Tabella tab_stats salvata in: {CSV_SUMMARY}")

In [ ]:
# Tab of Cesare's results
INPUT_CSV = "/content/drive/MyDrive/Colab Notebooks/Stats/RESULTS_CESARE/summaries_validation_results_qags.csv"
OUTPUT_CSV = "/content/drive/MyDrive/Colab Notebooks/Stats/tab_Cesare_stats.csv"
progress_to_test = {
    "1270": "Test1",
    "3627": "Test2",
    "4466": "Test3"
}

results_tab_stats = []

with open(INPUT_CSV, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)

    for row in reader:
        model = row['model']
        qags_score = float(row['qags_score'])
        mode = row['mode']
        progress_id = row['progress_id'].strip()
        test_name = progress_to_test.get(progress_id, mode)

        results_tab_stats.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "test": test_name,
            "model": model,
            "type": "TRUE/FALSE",
            "threshold": "",
            "success": qags_score * 100,
            "retrieval": ""
        })

with open(OUTPUT_CSV, mode="w", newline='', encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "Timestamp", "Test", "Modello", "Tipo Domande",
        "Threshold", "Successo (%)", "Retrieval Strategy"
    ])
    for result in results_tab_stats:
        writer.writerow([
            result["timestamp"],
            result["test"],
            result["model"],
            result["type"],
            result["threshold"],
            f"{result['success']:.2f}",
            result["retrieval"]
        ])

print(f"Tabella di statistiche salvata in: {OUTPUT_CSV}")